# Split HCB por Regional

Divide la matriz consolidada de HCB en archivos individuales por regional,
preservando fórmulas, formatos y eliminando hojas innecesarias.

### Lógica:
1. Lee la matriz fuente y extrae las regionales únicas (columna C).
2. Por cada regional: copia el archivo, elimina hojas innecesarias,
   copia los datos a una hoja temporal, reemplaza referencias `MATRIZ!`,
   filtra y elimina filas de otras regionales, renombra la hoja.
3. Genera un archivo `MATRIZ_{REGIONAL}.xlsx` por cada regional.

In [ ]:
import sys
import os
import time

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), 'src')))
from split_hcb_by_regional.split_regional import split_by_regional, get_regionals

# ==================================================================
# CONFIGURACIÓN
# ==================================================================
SOURCE_FILE = r"d:\ICBF\cost-tracking\data\replicacion hcb 5 junio\MATRIZ_2026_ADICION_COMUNITARIOS_MODFV1.xlsx"
OUTPUT_ROOT = r"d:\ICBF\cost-tracking\data\replicacion hcb 5 junio\REGIONALES"

# Opcional: filtrar solo regionales específicas (None = todas)
REGIONALES_FILTER = None  # ej. ["Antioquia", "Cundinamarca"]

# Contraseña para proteger hojas (solo MATRIZ quedará editable)
PASSWORD = "GMYC2026**"


## 1. Verificar regionales en la fuente

In [ ]:
t_start = time.time()
regionals = REGIONALES_FILTER or get_regionals(SOURCE_FILE)
print(f"Regionales encontradas: {len(regionals)}")
for i, r in enumerate(regionals, 1):
    print(f"  {i:2d}. {r}")
print(f"\nTiempo: {time.time()-t_start:.1f}s")

## 2. Ejecutar split por regional

In [ ]:
t_start = time.time()
created = split_by_regional(SOURCE_FILE, OUTPUT_ROOT, regionals=regionals, password=PASSWORD)
print(f"\nCompletado en {time.time()-t_start:.1f}s")
print(f"Archivos generados: {len(created)}")
for f in created:
    print(f"  {f}")

## 3. Validar resultado

In [ ]:
import win32com.client as win32

excel = win32.gencache.EnsureDispatch('Excel.Application')
excel.Visible = False
excel.DisplayAlerts = False

print(f"{'Regional':25s} {'Filas':>6s} {'Fórmulas':>9s} {'#REF!':>6s}")
print("-" * 50)

total_rows = 0
for f in created:
    wb = excel.Workbooks.Open(f)
    ws = wb.Sheets("MATRIZ")
    lr = ws.UsedRange.Rows.Count
    reg = ws.Cells(3, 3).Value or ""
    ref_errors = 0
    formulas = 0
    for r in range(3, lr + 1):
        for c in [5, 10, 18, 40, 47, 49, 58]:
            fml = ws.Cells(r, c).Formula
            if isinstance(fml, str):
                if "#REF!" in fml:
                    ref_errors += 1
                if fml.startswith("="):
                    formulas += 1
    total_rows += lr - 2
    print(f"{str(reg):25s} {lr-2:>6d} {formulas:>9d} {ref_errors:>6d}")
    wb.Close()

excel.Quit()
print("-" * 50)
print(f"{'TOTAL':25s} {total_rows:>6d}")